In [ ]:
import operator
from functools import reduce
from pathlib import Path, PurePath

import polars as pl
from tqdm import tqdm
import xlsxwriter

from hexagonal.files.spec import get_polars_dataframe

from rapidfuzz import fuzz

In [ ]:
import polars.selectors as cs

In [ ]:
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(100)

In [ ]:
def normaliser(colonne):
    if isinstance(colonne, str):
        colonne = pl.col(colonne)
    return (
        colonne
        .str.normalize("NFKD")  # splitter les diacritiques
        .str.to_lowercase()
        .str.replace_all(r"œ", "oe")    # à traiter spécifiquement en français
        .str.replace_all(r"\p{Nonspacing Mark}", "")  # retirer les diacritiques
        .str.replace_all(r"\pP", " ")  # ponctuation
        .str.replace_all(r"\s\s+", " ")
        .str.strip_chars()
    )

In [ ]:
def scorer(a):
    return float(fuzz.ratio(*a.values()))

# Conditions

In [ ]:
maire_soutien = (pl.col("parrainage_2022") == "MÉLENCHON Jean-Luc").fill_null(False)
maire_corse = pl.col("code_commune").str.slice(0, 2).is_in(["2A", "2B"])
maire_nouvelle_caledonie = pl.col("code_commune").str.slice(0, 3) == "988"
maire_outremer = pl.col("code_commune").str.slice(0, 2).is_in(["97", "98"])

# Sources de données

## COG

In [ ]:
communes = get_polars_dataframe("data/03_main/cog/communes.csv")

In [ ]:
departements = get_polars_dataframe("data/02_clean/cog/departements.csv")

## Annuaire

In [ ]:
mairies = get_polars_dataframe("data/02_clean/annuaire/mairies.csv").with_columns(
    pl.col("emails").str.split("\n|,", literal=False)
)

In [ ]:
conseils_departementaux = get_polars_dataframe("data/02_clean/annuaire/conseils_departementaux.csv").with_columns(
    code_departement=pl.when(pl.col("code_departement") == "67").then(pl.lit("6AE")).otherwise("code_departement")
).select("code_departement", "nom", cs.starts_with("postale_")).with_columns(
    mandat=pl.when(
        pl.col("code_departement").is_in([
            "975", 
            "977",
            "978",
            "987",
            "988",
        ])
    ).then(pl.lit("membre assemblée outremer"))
    .otherwise(pl.lit("conseiller départemental"))
)

In [ ]:
conseils_regionaux = get_polars_dataframe("data/02_clean/annuaire/conseils_regionaux.csv").select(
    "code_departement",
    "nom",
    cs.starts_with("postale_")
).join(
    departements.select("code_departement", "code_region"),
    on=["code_departement"],
    how="left",
).select(
    pl.all().exclude("code_departement")
).join(
    pl.concat([
        departements.select("code_region", "code_departement"),
        pl.DataFrame({
            "code_region": ["44", "84", "94"],
            "code_departement": ["6AE", "69M", "20"]
        })
    ]),
    on=["code_region"],
    how="left"
).with_columns(
    mandat=pl.when(
        pl.col("code_departement").is_in(["972", "973"])
    ).then(
        pl.lit("membre assemblée outremer")
    ).when(
        pl.col("code_departement") == "20"
    ).then(
        pl.lit("membre assemblée corse")
    ).otherwise(pl.lit("conseiller régional"))
)

## RNE

In [ ]:
nom_complet = pl.format(
    "{} {} {}",
    pl.col("sexe").replace_strict(["M", "F"], ["M.", "Mme"]),
    "prenom",
    "nom"
).alias("nom_complet")

In [ ]:
maires = get_polars_dataframe("data/02_clean/rne/conseillers_municipaux.csv").filter(
    pl.col("fonction").is_in(["Maire", "Maire délégué"])
).select(
    "code_commune", 
    "nom",
    "prenom",
    "sexe",
    pl.col("fonction").str.to_lowercase(),
    nom_complet,
).unique(
    ["code_commune", "nom_complet"], keep="first"
).sort(
    ["code_commune", "nom_complet", "fonction"]
).with_row_index().with_columns(
    pl.format("MAI{}", pl.col("index"))
)

In [ ]:
conseillers_departementaux = get_polars_dataframe("data/02_clean/rne/conseillers_departementaux.csv").unique(
    ["code_departement", "nom", "prenom", "sexe", "date_naissance"]
)

In [ ]:
conseillers_regionaux = get_polars_dataframe("data/02_clean/rne/conseillers_regionaux.csv").unique(
    ["code_departement", "nom", "prenom", "sexe", "date_naissance"]
)

In [ ]:
membres_csp = get_polars_dataframe("../../../data/02_clean/rne/conseillers_csp.csv").unique(
    ["code_csp", "nom", "prenom", "sexe", "date_naissance"]
)

## Parrainages

In [ ]:
parrainages = get_polars_dataframe("data/03_main/elections/2022-presidentielle-parrainages.csv").with_row_index().with_columns(
    index="CC" + pl.col("index").cast(pl.String()).str.zfill(5)
)

In [ ]:
parrainages_maires = parrainages.filter(
    pl.col("mandat").is_in(["maire", "maire délégué"])
).select(
    "index",
    pl.col("code_circonscription").alias("code_commune"),
    "nom",
    "prenom",
    "sexe",
    "candidat",
    pl.col("mandat").alias("fonction"),
    nom_complet
)

In [ ]:
parrainages_elus_territoriaux = parrainages.filter(
    pl.col("mandat").is_in(["conseiller départemental", "conseiller régional", "membre assemblée outremer", "membre assemblée corse", "conseiller métropole Lyon"])
).select(
    "index",
    pl.col("code_circonscription").replace(
        ["67", "68", "2A", "2B", "69M"],
        ["6AE", "6AE", "20", "20", "69"]
    ).alias("code_departement"),
    "mandat",
    "nom",
    "prenom",
    "sexe", 
    "candidat",
    nom_complet
)

In [ ]:
emails_individuels = pl.read_csv("out/2026-06-emails-maires.csv", schema_overrides={"code_commune": pl.String()}).with_columns(
    nom_complet
)

## Résultats électoraux

In [ ]:
presidentielle = get_polars_dataframe("data/02_clean/elections/2022-presidentielle-1-bureau_de_vote.parquet").group_by(
    ["code_commune", "numero_panneau"]
).agg(
    pl.col("nom").first(),
    pl.col("prenom").first(),
    pl.col("sexe").first(),
    pl.col("voix").sum()
).with_columns(
    part=(pl.col("voix") / pl.col("voix").sum().over(["code_commune"])).fill_nan(0.)
).sort(["code_commune", "numero_panneau"])

presidentielle_melenchon = presidentielle.filter(
    pl.col("nom") == "MÉLENCHON"
)

In [ ]:
europeenne = get_polars_dataframe("data/02_clean/elections/2024-europeenne-bureau_de_vote.parquet").group_by(
    ["code_commune", "numero_panneau"]
).agg(
    pl.col("nuance").first(),
    pl.col("liste_court").first(),
    pl.col("voix").sum()
).with_columns(
    part=(pl.col("voix") / pl.col("voix").sum().over(["code_commune"])).fill_nan(0.),
).sort(["code_commune", "numero_panneau"])

europeenne_lfi = europeenne.filter(
    pl.col("liste_court") == "LFI - UP"
)

In [ ]:
correspondances_municipales = get_polars_dataframe("../../../data/03_main/elections/2026-municipales-2-composition-listes-nominative.parquet")

In [ ]:
listes_municipales_t1 = get_polars_dataframe("../../../data/02_clean/elections/2026-municipales-1-candidats.parquet")
listes_municipales_t2 = get_polars_dataframe("../../../data/02_clean/elections/2026-municipales-2-candidats.parquet").join(
    correspondances_municipales,
    on=["code_commune", "numero_panneau", "ordre"]
).join(
    listes_municipales_t1.select("code_commune", "numero_panneau", "ordre", "nuance", "liste_court"),
    left_on=["code_commune", "numero_panneau_t1", "ordre_t1"],
    right_on=["code_commune", "numero_panneau", "ordre"],
    suffix="_t1"
)

In [ ]:
sieges_municipales_t1 = get_polars_dataframe("../../../data/02_clean/elections/2026-municipales-1-sieges.parquet").select(
    "code_commune", "numero_panneau", "elus_cm"
)
sieges_municipales_t2 = get_polars_dataframe("../../../data/02_clean/elections/2026-municipales-2-sieges.parquet").select(
    "code_commune", "numero_panneau", "elus_cm"
)

In [ ]:
elus_t1 = listes_municipales_t1.join(
    sieges_municipales_t1,
    on=["code_commune", "numero_panneau"],
    how="inner"
).filter(pl.col("ordre") <= pl.col("elus_cm"))

elus_t2 = listes_municipales_t2.join(
    sieges_municipales_t2,
    on=["code_commune", "numero_panneau"],
    how="inner"
).filter(pl.col("ordre") <= pl.col("elus_cm"))

In [ ]:
nuances_elus_municipaux = pl.concat([
    elus_t1.select(
        "code_commune", "nom", "prenom", "sexe", pl.col("nuance").alias("nuance_t1"), pl.col("liste_court").alias("liste_t1")
    ),
    elus_t2.select(
        "code_commune", "nom", "prenom", "sexe", "nuance_t1", pl.col("liste_court_t1").alias("liste_t1")
    )
])

## Opérateurs splink

In [ ]:
import splink.exploratory as exp
import splink.comparison_level_library as cll
import splink.comparison_library as cl
from splink import DuckDBAPI, block_on, SettingsCreator, Linker

In [ ]:
full_name_comparison = {
    "output_column_name": "Nom complet",
    "comparison_levels": [
        {
            "sql_condition": "(prenom_l IS NULL OR prenom_r IS NULL) and (nom_l IS NULL OR nom_r IS NULL)",
            "label_for_charts": "Null",
            "is_null_level": True,
        },
        cll.ExactMatchLevel("nom_complet", term_frequency_adjustments=True),
        cll.JaroWinklerLevel("nom_complet", 0.95),
        cll.ColumnsReversedLevel("nom", "prenom"),
        {
            "sql_condition": "jaro_winkler_similarity(prenom_l, nom_r) + jaro_winkler_similarity(nom_l, prenom_r) >= 1.8",
            "label_for_charts": "switched + jaro_winkler_similarity >= 1.8",
        },
        {
            "sql_condition": """
                (
                    contains(prenom_l, prenom_r) or contains(prenom_r, prenom_l)
                ) and (
                    contains(nom_l, nom_r) or contains(nom_r, nom_l)
                )""",
            "label_for_charts": "contained"
        },
        cll.ElseLevel(),
    ]
}

code_commune_comparison = {
    "output_column_name": "Commune",
    "comparison_levels": [
        {
            "sql_condition": "code_commune_l IS NULL AND code_commune_r IS NULL",
            "label_for_charts": "Null",
            "is_null_level": True,
        },
        cll.ExactMatchLevel("code_commune"),
        cll.ExactMatchLevel("code_departement", term_frequency_adjustments=True),
        cll.ElseLevel(),
    ]
}

# Préparer les données

## Identifier les maires qui ont été parrains en 2022

In [ ]:
exclusions = [
    ("17114", "Mme Marie-Noëlle SURAUD"),
    ("55100", "M. Loïc VINCENT"),
    ("61225", "M. Cyrille COTREL-LASSAUSSAYE"),
    ("30129", "M. Thierry ORTIZ"),
    ("61286", "Mme Marie-Claude CORVÉ"),
]

In [ ]:
maires_qualifies = maires.join(
    parrainages_maires.select(
        pl.col("index").alias("index_parrainage"),
        "code_commune",
        "fonction",
        "nom_complet",
        "candidat"), 
    on=["code_commune"],
    how="left",
    suffix="_parrainage"
).with_columns(
    score=pl.when(
        pl.col("nom_complet_parrainage").is_not_null()
    ).then(
        pl.struct(normaliser("nom_complet"), normaliser("nom_complet_parrainage")).map_elements(
            scorer,
            return_dtype=pl.Float64(),
            skip_nulls=True,
        )
    )
).sort(
    ["code_commune", "nom_complet", "score"],
    descending=[False, False, True]
).unique(
    ["code_commune", "nom_complet"]
).select(
    *maires.columns,
    "index_parrainage",
    parrainage_2022=pl.when(
        (pl.col("score") >= 80.) & ~pl.struct("code_commune", "nom_complet").is_in(
            pl.Series(exclusions, dtype=pl.Struct({"code_commune": pl.String(), "nom_complet": pl.String()})).implode()
        )
    ).then("candidat")
)

In [ ]:
noms = [
    normaliser("prenom"),
    normaliser("nom"),
    pl.format("{} {}", normaliser("prenom"), normaliser("nom")).alias("nom_complet")
]

code_departement = pl.when(
    pl.col("code_commune").str.slice(0, 2).is_in(["97", "98"])
).then(
    pl.col("code_commune").str.slice(0, 3)
).otherwise(
    pl.col("code_commune").str.slice(0, 2)
).alias("code_departement")

In [ ]:
api = DuckDBAPI()
api.register_multiple_tables(
    [
        maires.select("index", "code_commune", code_departement, *noms, "fonction", "sexe"),
        parrainages_maires.select("index", "code_commune", code_departement, *noms, "fonction", "sexe")
    ],
    ["rne", "parrainages"],
)

blocks = [
    block_on("prenom"),
    block_on("nom"),
    block_on("code_commune"),
    block_on("code_departement")
]

comparisons = [
    full_name_comparison,
    code_commune_comparison,
    cl.ExactMatch("sexe").configure(term_frequency_adjustments=True),    
]

settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="index",
    blocking_rules_to_generate_predictions=blocks,
    comparisons=comparisons,
    retain_intermediate_calculation_columns=True,
)

maires_linker = Linker(
    ["rne", "parrainages"],
    settings,
    api
)

In [ ]:
deterministic_rules = [
    block_on("nom", "prenom", "sexe"),
]

maires_linker.training.estimate_probability_two_random_records_match(
    deterministic_rules,
    recall=0.90
)

In [ ]:
maires_linker.training.estimate_u_using_random_sampling(1e7)

In [ ]:
session_departement = maires_linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("code_departement"), estimate_without_term_frequencies=True
)

In [ ]:
session_prenom = maires_linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("prenom"), estimate_without_term_frequencies=True
)

In [ ]:
predictions = maires_linker.inference.predict(threshold_match_probability=0.2)

In [ ]:
correspondances_maires = predictions.as_duckdbpyrelation().pl().filter(
    pl.col("match_probability") > .95
).select(
    index_rne="index_l",
    index_parrainage="index_r",
)

In [ ]:
maires_qualifies = maires.join(
    correspondances_maires,
    left_on="index",
    right_on="index_rne",
    how="left"
).join(
    parrainages_maires.select(
        pl.col("index").alias("index_parrainage"),
        pl.col("candidat").alias("parrainage_2022"),
    ),
    on="index_parrainage",
    how="left",
)

## Identifier les conseillers départementaux, conseillers régionaux et conseillers territoriaux parrains en 2022

In [ ]:
elus_territoriaux = pl.concat(
    [
        conseillers_departementaux.select(
            "code_departement",
            "nom",
            "prenom",
            "sexe",
            pl.lit("conseiller départemental").alias("mandat")
        ),
        conseillers_regionaux.select(
            pl.col("code_departement").replace(["69M"], ["69"]),
            "nom",
            "prenom",
            "sexe",
            pl.lit("conseiller régional").alias("mandat")
        ),
        membres_csp.filter(pl.col("code_csp") != "988C").select(
            pl.col("code_csp").str.slice(0, pl.col("code_csp").str.len_chars() - 1).alias("code_departement"),
            "nom",
            "prenom",
            "sexe",
            pl.when(
                pl.col("code_csp") == "20R"
            ).then(
                pl.lit("membre assemblée corse")
            ).when(
                pl.col("code_csp") == "69M"
            ).then(
                pl.lit("conseiller métropole Lyon")
            ).otherwise(pl.lit("membre assemblée outremer")).alias("mandat")
        )
    ]
).with_row_index().with_columns(index="RNE" + pl.col("index").cast(pl.String()).str.zfill(4))

In [ ]:
noms = [
    normaliser("prenom"),
    normaliser("nom"),
    pl.format("{} {}", normaliser("prenom"), normaliser("nom")).alias("nom_complet")
]

In [ ]:
api = DuckDBAPI()
api.register_multiple_tables(
    [
        elus_territoriaux.with_columns(*noms), 
        parrainages_elus_territoriaux.select(
            "index",
            "code_departement",
            *noms,
            "sexe",
            "mandat"
        )
    ],
    ["rne", "parrainages"]
)

blocks = [
    block_on("prenom", "nom"),
    block_on("code_departement"),
    block_on("mandat"),
]

comparisons = [
    full_name_comparison,
    cl.ExactMatch("sexe").configure(term_frequency_adjustments=True),
    cl.ExactMatch("code_departement").configure(term_frequency_adjustments=True),
    cl.ExactMatch("mandat")
]

settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="index",
    blocking_rules_to_generate_predictions=blocks,
    comparisons=comparisons,
    retain_intermediate_calculation_columns=True,
)

elus_territoriaux_linker = Linker(
    ["rne", "parrainages"],
    settings,
    api,
)

In [ ]:
deterministic_rules = [
    block_on("nom", "prenom", "sexe"),
]

elus_territoriaux_linker.training.estimate_probability_two_random_records_match(
    deterministic_rules,
    recall=0.95
)

In [ ]:
elus_territoriaux_linker.training.estimate_u_using_random_sampling(1e7)

In [ ]:
session_mandat = elus_territoriaux_linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("mandat"), estimate_without_term_frequencies=True
)

In [ ]:
session_departement = elus_territoriaux_linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("code_departement"), estimate_without_term_frequencies=True
)

In [ ]:
predictions = elus_territoriaux_linker.inference.predict(threshold_match_probability=0.2)

In [ ]:
correspondances_elus_territoriaux = predictions.as_duckdbpyrelation().pl().filter(
    pl.col("match_probability") > .95
).select(
    index_rne="index_l",
    index_parrainage="index_r",
)

In [ ]:
elus_parrainage = elus_territoriaux.join(
    correspondances_elus_territoriaux,
    left_on=["index"],
    right_on=["index_rne"],
    how="left",
    maintain_order="left"
).join(
    parrainages_elus_territoriaux.select(
        "index",
        "candidat",
    ),
    left_on=["index_parrainage"],
    right_on=["index"],
    how="left",
    maintain_order="left",
).select(
    "index",
    "code_departement",
    "mandat",
    "nom",
    "prenom",
    nom_complet,
    "sexe",
    pl.col("candidat").alias("parrainage_2022")
)

## Identifier la mairie principale

In [ ]:
dedup = [
    "d268cedd-dc8c-4204-9226-ef3ff576d91d",
    "6cf098c7-53d0-4e19-9d95-15b0bc89d3e9",
    "873a2bf8-aaec-4c1e-a0bb-177c74b0a2e4",
    "b013c5c8-7f0c-4afe-9052-6a0233092c38",
    "15944e99-e55a-475a-8132-5e160995799e",
    "deae784b-d2c9-4527-aee0-55ddf2939288",
    "ca05a12a-7a3a-475a-9d75-3945467733aa",
    "2ee8945b-8707-479d-b9a9-f0235df112f1",
    "3e52c219-b9f4-4150-b517-f1bc0a5f4e41",
    "8bb65ac2-f71f-4ddf-942d-1a363cb32983",
    "34f09172-217b-4c03-932b-cbf47f2117c2",
    "568a88ff-21dc-4d10-b166-fdcc6086bfd9",
    "114a1668-d6d8-4809-b8f6-2f5e29acfcab",
    "af20c30b-5cc7-4b1e-b7f6-65d766a4b503"
]

In [ ]:
sans_mairie_deleguee = ~pl.col("nom").str.contains(r"(?i)(déléguéé?e|annexe)")
unique = pl.col("nom").count().over("code_commune") == 1
nom_correspond = normaliser("nom").str.contains("^mairie " + normaliser("physique_commune")+"$")
in_dedup = pl.col("id").is_in(dedup)

In [ ]:
mairies_principales = mairies.filter(
    sans_mairie_deleguee
).filter(
    unique | in_dedup | (nom_correspond & ~in_dedup.any().over("code_commune"))
).sort("code_commune")

## Traiter les conseils départementaux et régionaux

In [ ]:
additionnels = [
    {
        "code_departement": "986",
        "nom": "Assemblée territoriale de Wallis-et-Futuna",
        "postale_service_distribution": "BP 31 Mata'Utu - Havelu Hahake",
        "postale_code_postal": "98600",
        "postale_commune": "Uvea",
        "mandat": "membre assemblée outremer"
    },
    {
        "code_departement": "987",
        "nom": "Assemblée de la Polynésie française",
        "postale_service_distribution": "BP 28",
        "postale_commune": "Papeete",
        "postale_code_postal": "98713",
        "mandat": "membre assemblée outremer",
    }
]

In [ ]:
conseils = pl.concat(
    [
        conseils_departementaux,
        conseils_regionaux.select("code_departement", "nom", cs.starts_with("postale_"), "mandat"),
        pl.from_records(additionnels, orient="row"),
    ],
    how="diagonal"
).unique(["code_departement", "mandat"], keep="last")

## Vérifier qu'on a bien tout

In [ ]:
maires_qualifies.filter(~pl.col("code_commune").is_in(mairies_principales["code_commune"].implode()))

# Préparer les courriers papier

## Maires

In [ ]:
champ_lieu = (
    pl.when(
        pl.col("code_commune").str.slice(0, 2).is_in(["2A", "2B"])
    ).then(
        pl.lit("corse")
    ).when(
        pl.col("code_commune").str.slice(0, 3) == "988"
    ).then(
        pl.lit("nouvelle-caledonie")
    ).when(
        pl.col("code_commune").str.slice(0, 2).is_in(["97", "98"])
    ).then(
        pl.lit("outremer")
    )
)

champ_parrains = (
    pl.when(
        pl.col("parrainage_2022") == "MÉLENCHON Jean-Luc"
    ).then(
        pl.lit("parrains")
    )
)

In [ ]:
adresses = maires_qualifies.select(
    "code_commune", "nom_complet", "sexe", "fonction", "parrainage_2022"
).join(
    mairies_principales.select(
        "code_commune",
        pl.col("nom").alias("postale_nom"),
        *(c for c in mairies_principales.columns if c.startswith("postale_"))
    ),
    on=["code_commune"],
    how="inner"
).with_columns(
    courrier=pl.concat_str(
        pl.lit("maires"),
        champ_lieu,
        champ_parrains,
        separator="-",
        ignore_nulls=True,
    )
)

In [ ]:
adresses["courrier"].value_counts(sort=True)

In [ ]:
formes_mairie = ["mairie", "hotel de ville"]
colonnes_adresses = ["postale_numero_voie", "postale_complement1", "postale_complement2", "postale_service_distribution"]

adresse_contient_mairie = reduce(
    operator.or_, (normaliser(c).is_in(formes_mairie) for c in colonnes_adresses)
)

parties_adresse = pl.concat_list(
    *(pl.col(c).str.to_uppercase() for c in colonnes_adresses),
    pl.format("{} {}", "postale_code_postal", "postale_commune").str.to_uppercase(),
).list.filter(pl.element().is_not_null())

adresse = pl.when(
    adresse_contient_mairie
).then(
    pl.format("{}\n{}", pl.col("nom_complet").str.to_uppercase(), parties_adresse.list.join("\n")),
).otherwise(
    pl.format("{}\nHÔTEL DE VILLE\n{}", pl.col("nom_complet").str.to_uppercase(), parties_adresse.list.join("\n"))
).alias("adresse_complete")

In [ ]:
adresses.filter(pl.col("courrier") != "maires").select(
    pl.col("code_commune").alias("code"),
    "nom_complet",
    "sexe",
    "courrier",
    adresse
).write_csv("out/adresses_maires_speciaux.csv")

## Autres élus

In [ ]:
lieux_speciaux = {
    "20": "corse",
    "971": "outremer",
    "972": "martinique",
    "973": "guyane",
    "974": "outremer",
    "975": "saint-pierre-et-miquelon",
    "976": "outremer",
    "977": "saint-barthelemy",
    "978": "saint-martin",
    "986": "wallis-et-futuna",
    "987": "polynesie",
    "988": "nouvelle-caledonie",
}

In [ ]:
conseils.filter(pl.col("code_departement") == "971")

In [ ]:
courriers_elus_territoriaux = elus_parrainage.join(
    conseils,
    on=["code_departement", "mandat"],
    how="left"
).with_columns(
    courrier=pl.concat_str(
        [
            pl.col("mandat").replace_strict(
                ["conseiller départemental", "conseiller régional", "membre assemblée outremer", "membre assemblée corse"],
                ["cd", "cr", "membres", "membres"]
            ),
            pl.col("code_departement").replace_strict(
                list(lieux_speciaux.keys()),
                list(lieux_speciaux.values()),
                default=pl.lit(None),
            ),
            pl.when(pl.col("parrainage_2022") == "MÉLENCHON Jean-Luc").then(pl.lit("parrains"))
        ],
        separator="-",
        ignore_nulls=True
    )
).select(
    pl.col("code_departement").alias("code"),
    "nom_complet",
    "sexe",
    "courrier",
    pl.concat_str(
        pl.col("nom_complet").str.to_uppercase(),
        pl.col("nom").str.to_uppercase(),
        pl.col("postale_numero_voie").str.to_uppercase(),
        pl.col("postale_complement1").str.to_uppercase(),
        pl.col("postale_complement2").str.to_uppercase(),
        pl.col("postale_service_distribution").str.to_uppercase(),
        pl.format("{} {}", "postale_code_postal", "postale_commune").str.to_uppercase(),
        separator="\n",
        ignore_nulls=True,
    ).alias("adresse_complete"),
)

In [ ]:
courriers_elus_territoriaux.write_csv("out/adresses_territoriaux.csv")

# Préparer les envois emails

In [ ]:
emails_mairies = mairies_principales.select(
    "code_commune", "emails"
).filter(pl.col("emails").is_not_null())

In [ ]:
envois_mairies = maires_qualifies.join(
    emails_mairies,
    on=["code_commune"],
).sort(["code_commune", "fonction"]).explode("emails").unique(["emails"], keep="first")

In [ ]:
envois_individuels = maires_qualifies.join(
    emails_individuels,
    on=["code_commune"],
    suffix="_email"
).with_columns(
    score=pl.struct(normaliser("nom_complet"), normaliser("nom_complet_email")).map_elements(
        lambda s: fuzz.token_ratio(*s.values()),
        return_dtype=pl.Float64(),
        skip_nulls=True,
    )
).filter(pl.col("score") == 100.)

In [ ]:
envois_emails = pl.concat([
    envois_mairies.select(
        "code_commune", "nom", "prenom", "sexe", "fonction", "parrainage_2022", pl.col("emails").alias("email")
    ),
    envois_individuels.select(
        "code_commune", "nom", "prenom", "sexe", "fonction", "parrainage_2022", "email"
    )
])

In [ ]:
def pour_envoi(emails, chemin):
    emails.select(
        "email",
        pl.format("{prenom} {nom}").alias("name"),
        pl.struct("code_commune", "fonction", "parrainage_2022", "sexe").struct.json_encode().alias("attributes")
    ).write_csv(chemin) 

In [ ]:
echantillons = {
    "corse": pl.col("code_commune").str.slice(0, 2).is_in(["2A", "2B"]),
    "nouvelle-calédonie": pl.col("code_commune").str.slice(0, 3) == "988",
    "outremer": pl.col("code_commune").str.slice(0, 2) == "97",
}

echantillons["hexagone"] = ~reduce(operator.or_, echantillons.values())

In [ ]:
for nom, cond in echantillons.items():
    pour_envoi(
        envois_emails.filter(cond),
        f"emails_{nom}.csv"
    )

# Extraction pour les référents

In [ ]:
details_mairies = mairies_principales.select(
    "code_commune",
    pl.col("telephone").alias("Téléphone mairie"),
    pl.col("emails").list.join(" / ").alias("Email mairie"),
    pl.concat_str(
        "physique_numero_voie",
        "physique_complement1",
        "physique_complement2",
        separator=" - ",
        ignore_nulls=True,
    ).alias("Adresse mairie"),
    pl.col("ouverture").str.replace_all(r"\n", ", ").alias("Horaires mairie")
)

In [ ]:
maires_extrac = maires_qualifies.join(
    communes.select("code_commune", pl.col("nom").alias("Commune")),
    on=["code_commune"],
    how="left",
).join(
    europeenne_lfi.select("code_commune", pl.col("part").alias("LFI euro24")),
    on=["code_commune"],
    how="left",
).join(
    presidentielle_melenchon.select("code_commune", pl.col("part").alias("Mélenchon prés22")),
    on=["code_commune"],
    how="left",
).join(
    nuances_elus_municipaux,
    on=["code_commune", "nom", "prenom"],
    how="left",
).join(
    details_mairies,
    on=["code_commune"],
    how="left"
).select(
    code_departement,
    "code_commune",
    "Commune",
    pl.col("fonction").alias("mandat"),
    "nom",
    "prenom",
    "sexe",
    "parrainage_2022",
    "LFI euro24",
    "Mélenchon prés22",
    pl.col("liste_t1").alias("Nom de la liste aux municipales 26"),
    pl.col("nuance_t1").alias("Nuance de la liste aux municipales 26"),
    "Téléphone mairie",
    "Email mairie",
    "Adresse mairie",
    "Horaires mairie",
)

In [ ]:
renommage = {
    "code_departement": "Département",
    "code_commune": "Code INSEE",
    "nom": "Nom",
    "prenom": "Prénom",
    "sexe": "Sexe",
    "mandat": "Mandat",
    "parrainage_2022": "Parrainage 2022"
}

In [ ]:
tous_extrac = pl.concat(
    [
        maires_extrac.with_columns(
            code_departement,
        ),
        elus_parrainage.select(
            "code_departement",
            "nom",
            "mandat",
            "prenom",
            "sexe",
            "parrainage_2022",
        )
    ],
    how="diagonal"
).rename(renommage).with_columns(
    fichier=pl.when(
        pl.col("Département").is_in(["20", "2A", "2B"]),
    ).then(
        pl.lit("20")
    ).when(
        pl.col("Département").is_in(["67", "68", "6AE"]),
    ).then(
        pl.lit("6AE")
    ).otherwise(
        "Département"
    )
).sort(["Département", "Code INSEE", "Nom", "Prénom"])

In [ ]:
widths = {
    "Département": 100,
    "Code INSEE": 70,
    "Commune": 160,
    "Nom": 180,
    "Prénom": 150,
    "Sexe": 50,
    "Mandat": 160,
    "Parrainage 2022": 160,
    "LFI euro24": 100,
    "Mélenchon prés22": 100,
    "Nom de la liste aux municipales 26": 200,
    "Nuance de la liste aux municipales 26": 170,
    "Téléphone mairie": 100,
    "Email mairie": 150,
    "Adresse mairie": 300,
    "Horaires mairie": 400
}

In [ ]:
header_format = {
    "bold": True,
    "text_wrap": True,
}

percent_format = '0.00" "%'

column_formats = {
    "LFI euro24": percent_format,
    "Mélenchon prés22": percent_format,
}

In [ ]:
for fichier in tqdm(tous_extrac["fichier"].unique()):
    with xlsxwriter.Workbook(Path("out/extraction") / f"{fichier}.xlsx") as workbook:
        worksheet = workbook.add_worksheet()
        tous_extrac.filter(
            pl.col("fichier") == fichier
        ).write_excel(
            workbook=workbook,
            worksheet=worksheet,
            header_format=header_format,
            column_formats=column_formats,
            column_widths=widths,
        )
        
        worksheet.set_row(0, 32)

In [ ]:
elus_parrainage

In [ ]:
elus_parrainage.filter(pl.col("parrainage_2022") == "MÉLENCHON Jean-Luc").filter(pl.col("code_departement").str.len_chars() >= 3).select(pl.struct("code_departement", "mandat").value_counts())

In [ ]:
from paramiko.client import SSHClient

client = SSHClient()
client.load_host_keys("/home/salome/Projets/playbooks/ssh_known_hosts")
client.connect("212.47.232.89", username="root")

sftp = client.open_sftp()

root = PurePath("/mnt/storage/data/__groupfolders/8/files")

for f in Path("out/extraction").glob("*.xlsx"):
    dep = f.stem
    d = root / dep

    sftp.put(str(f), str(d / "liste_elus.xlsx"))